In [1]:
from dotenv import load_dotenv

# Load environment variables from .env
load_dotenv()

True

In [9]:
from typing import TypedDict

from langchain.agents.middleware import dynamic_prompt
from langgraph.checkpoint.memory import InMemorySaver


# Runtime context schema
class RuntimeContext(TypedDict):
    user_id: str
    writing_goal: str
    tone: str


# Dynamic system prompt that uses runtime context per request
@dynamic_prompt
def persona_prompt(request) -> str:
    ctx: RuntimeContext = request.runtime.context
    return (
        "You are an assistant for Inkwell, a markdown writing studio for developer-writers.\n"
        f"User: {ctx['user_id']}\n"
        f"Writing goal: {ctx['writing_goal']}\n"
        f"Preferred tone: {ctx['tone']}\n"
        "Give practical, short, implementation-ready guidance."
    )


# In-memory checkpointing for multi-turn continuity
memory = InMemorySaver()

In [10]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

contextual_agent = create_agent(
    model="claude-haiku-4-5",
    middleware=[persona_prompt],
    context_schema=RuntimeContext,
    checkpointer=memory,
)

runtime_context: RuntimeContext = {
    "user_id": "linnie",
    "writing_goal": "ship an article editor with FastAPI + React",
    "tone": "concise",
}

thread_cfg = {"configurable": {"thread_id": "inkwell-session-1"}}

In [11]:
# Turn 1
r1 = contextual_agent.invoke(
    {"messages": [HumanMessage(content="Draft a 3-step plan to add autosave to Inkwell.")]},
    context=runtime_context,
    config=thread_cfg,
)
print("Turn 1:", r1["messages"][-1].content)

Turn 1: # Autosave Implementation Plan

## 1. Backend: Add a Draft Endpoint
Create a FastAPI endpoint to persist draft states without validation:

```python
@app.post("/articles/{article_id}/drafts")
async def save_draft(article_id: str, body: dict):
    # Store raw content + metadata (title, body, timestamp)
    # Use upsert to overwrite previous draft
    db.drafts.update_one(
        {"article_id": article_id},
        {"$set": body},
        upsert=True
    )
    return {"saved_at": datetime.now()}
```

**Key: Bypass full validation.** Drafts should accept incomplete content.

---

## 2. Frontend: Debounce & Track Changes
Add autosave to your React editor with debouncing:

```javascript
const [content, setContent] = useState("");
const debouncedSave = useCallback(
  debounce((data) => {
    fetch(`/articles/${id}/drafts`, {
      method: "POST",
      body: JSON.stringify(data)
    });
  }, 2000),
  [id]
);

const handleChange = (e) => {
  setContent(e.target.value);
  debouncedSav

In [12]:
# Turn 2 (same thread_id => state restored via InMemorySaver)
r2 = contextual_agent.invoke(
    {
        "messages": [
            HumanMessage(content="Now turn that into a checklist with acceptance criteria.")
        ]
    },
    context=runtime_context,
    config=thread_cfg,
)
print("Turn 2:", r2["messages"][-1].content)

Turn 2: # Autosave Checklist

## 1. Backend: Draft Endpoint
- [ ] Create `POST /articles/{article_id}/drafts` endpoint
  - **Accepts:** `title`, `body`, optional metadata
  - **Returns:** `{"saved_at": timestamp}`
  - **No validation errors** — incomplete drafts allowed
- [ ] Set up MongoDB collection for drafts (or equivalent)
  - **Upsert logic:** overwrites previous draft for same `article_id`
  - **Indexes:** `article_id` for fast lookups
- [ ] Test with curl/Postman: save incomplete content, verify upsert works

---

## 2. Frontend: Debounced Autosave
- [ ] Add debounce utility (lodash or custom)
- [ ] Hook autosave to editor `onChange`:
  - **Debounce delay:** 2 seconds
  - **Payload:** `{title, body}`
  - **No UI feedback on every keystroke** (avoid flashing)
- [ ] Test: type → wait 2s → verify network request fires once (not per keystroke)
- [ ] Handle fetch errors gracefully (silent fail or toast)

---

## 3. Draft Recovery
- [ ] Create `GET /articles/{article_id}/draft` endpo